ChEMBL Database
The ChEMBL Database is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications. [Data as of March 25, 2020; ChEMBL version 26].

Installing libraries
Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [5]:
import sys
!{sys.executable} -m pip install chembl_webresource_client


   -------- ------------------------------- 1/5 [url-normalize]
   ---------------- ----------------------- 2/5 [cattrs]
   ---------------- ----------------------- 2/5 [cattrs]
   ---------------- ----------------------- 2/5 [cattrs]
   ---------------- ----------------------- 2/5 [cattrs]
   ---------------- ----------------------- 2/5 [cattrs]
   ---------------- ----------------------- 2/5 [cattrs]
   ------------------------ --------------- 3/5 [requests-cache]
   ------------------------ --------------- 3/5 [requests-cache]
   ------------------------ --------------- 3/5 [requests-cache]
   -------------------------------- ------- 4/5 [chembl_webresource_client]
   -------------------------------- ------- 4/5 [chembl_webresource_client]
   -------------------------------- ------- 4/5 [chembl_webresource_client]
   -------------------------------- ------- 4/5 [chembl_webresource_client]
   ---------------------------------------- 5/5 [chembl_webresource_client]



In [1]:
# Importing the necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

In [2]:
import chembl_webresource_client
print(chembl_webresource_client.__version__)

development


In [3]:
import pandas as pd
print(pd.__version__)

3.0.3


In [4]:
# Targeting the BACE 1 enzyme
target = new_client.target
target_query = list(target.search('BACE1'))
targets= pd.DataFrame.from_dict(target_query)
targets


,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Cricetulus griseus,Beta-secretase 1,14.0,False,CHEMBL3297642,"[{'accession': 'G3IAK4', 'component_descriptio...",SINGLE PROTEIN,10029
1,[],Homo sapiens,Beta-secretase 1,11.0,False,CHEMBL4822,"[{'accession': 'P56817', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Mus musculus,Beta-secretase 1,11.0,False,CHEMBL4593,"[{'accession': 'P56818', 'component_descriptio...",SINGLE PROTEIN,10090
3,[],Rattus norvegicus,Beta-secretase 1,11.0,False,CHEMBL3259473,"[{'accession': 'P56819', 'component_descriptio...",SINGLE PROTEIN,10116
4,[],Homo sapiens,Beta-secretase (BACE),7.0,False,CHEMBL2111390,"[{'accession': 'Q9Y5Z0', 'component_descriptio...",PROTEIN FAMILY,9606


In [5]:
selected_target = targets.target_chembl_id[1]
selected_target

'CHEMBL4822'

In [6]:
activity = new_client.activity
res= activity.filter(target_chembl_id='CHEMBL4822',standard_type="IC50").only(['molecule_chembl_id','canonical_smiles','standard_value'])



In [ ]:
# # For checking how many records are there in the result set

# count = 0
# for _ in res:
#     count += 1
#     if count % 1000 == 0:
#         print(count)

In [12]:

# df = pd.DataFrame.from_dict(res)
import time
import requests
# from urllib3.exceptions import 
from urllib3.exceptions import ProtocolError
from http.client import RemoteDisconnected

for attempt in range(5):
    try:
        df = pd.DataFrame(list(res))
        break
    except (requests.exceptions.ConnectionError,ProtocolError, RemoteDisconnected) as e:
        print(f"Attempt {attempt + 1} failed: {e}. Retrying in 5 seconds...")
        time.sleep(5)
# df = pd.DataFrame(list(res))


In [13]:
df

,canonical_smiles,molecule_chembl_id,standard_value,value
0,CC(C)C[C@H](NC(=O)[C@@H](NC(=O)[C@@H](N)CCC(=O...,CHEMBL406146,413.0,413.0
1,CC(C)C[C@H](NC(=O)[C@H](CC(N)=O)NC(=O)[C@@H](N...,CHEMBL78946,2.0,0.002
2,CCC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(C)=O)[C@@H]...,CHEMBL324109,460.0,0.46
3,CC(=O)NCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)CC(=O)...,CHEMBL114147,9000.0,9.0
4,CC(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](Cc1ccccc1...,CHEMBL419949,5600.0,5.6
...,...,...,...,...
13772,Fc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)c(F)c1,CHEMBL4759108,210.0,0.21
13773,COc1cc(/C=C/c2nnc(-c3ccc(OC(F)(F)F)cc3)o2)ccc1O,CHEMBL5197518,211.0,0.211
13774,COc1cc2oc(C(=O)Nc3cccc(CN(C)Cc4ccccc4)c3)cc(=O...,CHEMBL5202021,6700.0,6.7
13775,Nc1c2c(nc3c1C(c1cccc([N+](=O)[O-])c1)C1=C(O3)C...,CHEMBL6188525,19600.0,19.6


In [14]:
df.to_csv('BACE1_activities_raw.csv', index=False)

In [15]:
# Handling missing data
df2 = df[df['standard_value'].notna()]
df2=df2[df.canonical_smiles.notna()]


C:\Users\anuja\AppData\Local\Temp\ipykernel_23652\2693991145.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2=df2[df.canonical_smiles.notna()]


In [16]:
len(df2.canonical_smiles.unique())

8815

In [17]:
df3 = df2.drop_duplicates(['canonical_smiles'])
df3

,canonical_smiles,molecule_chembl_id,standard_value,value
0,CC(C)C[C@H](NC(=O)[C@@H](NC(=O)[C@@H](N)CCC(=O...,CHEMBL406146,413.0,413.0
1,CC(C)C[C@H](NC(=O)[C@H](CC(N)=O)NC(=O)[C@@H](N...,CHEMBL78946,2.0,0.002
2,CCC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(C)=O)[C@@H]...,CHEMBL324109,460.0,0.46
3,CC(=O)NCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)CC(=O)...,CHEMBL114147,9000.0,9.0
4,CC(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](Cc1ccccc1...,CHEMBL419949,5600.0,5.6
...,...,...,...,...
13769,COC(=O)c1ccc(/C=N/NC(=O)c2ccc(Cl)nc2)cc1,CHEMBL6172404,2800.0,2.8
13770,COc1cc(/C=N/NC(=O)c2ccc(Cl)nc2)ccc1O,CHEMBL6165462,600.0,0.6
13771,Clc1ccc(-c2nnc(CNC3CCN(Cc4ccccc4)CC3)o2)c(Cl)c1,CHEMBL6161637,250.0,0.25
13775,Nc1c2c(nc3c1C(c1cccc([N+](=O)[O-])c1)C1=C(O3)C...,CHEMBL6188525,19600.0,19.6


In [18]:
# <!-- save the dataframe to CSV file -->
df3.to_csv('BACE1_activities_cleaned.csv', index=False)


Labeling compounds as either being active, inactive or intermediate
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be active while those greater than 10,000 nM will be considered to be inactive. As for those values in between 1,000 and 10,000 nM will be referred to as intermediate.

In [19]:
df4 = pd.read_csv('BACE1_activities_cleaned.csv')


In [20]:
bioactivity_threshold =[]
for i in df4.standard_value:
  if float(i) >= 10000:
    bioactivity_threshold.append("inactive")
  elif float(i) <= 1000:
    bioactivity_threshold.append("active")
  else:
    bioactivity_threshold.append("intermediate")
    

In [21]:
bioactivity_class =pd.Series(bioactivity_threshold, name='class')
df5 = pd.concat([df4,bioactivity_class], axis=1)
df5

,canonical_smiles,molecule_chembl_id,standard_value,value,class
0,CC(C)C[C@H](NC(=O)[C@@H](NC(=O)[C@@H](N)CCC(=O...,CHEMBL406146,413.0,413.000,active
1,CC(C)C[C@H](NC(=O)[C@H](CC(N)=O)NC(=O)[C@@H](N...,CHEMBL78946,2.0,0.002,active
2,CCC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(C)=O)[C@@H]...,CHEMBL324109,460.0,0.460,active
3,CC(=O)NCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)CC(=O)...,CHEMBL114147,9000.0,9.000,intermediate
4,CC(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](Cc1ccccc1...,CHEMBL419949,5600.0,5.600,intermediate
...,...,...,...,...,...
8810,COC(=O)c1ccc(/C=N/NC(=O)c2ccc(Cl)nc2)cc1,CHEMBL6172404,2800.0,2.800,intermediate
8811,COc1cc(/C=N/NC(=O)c2ccc(Cl)nc2)ccc1O,CHEMBL6165462,600.0,0.600,active
8812,Clc1ccc(-c2nnc(CNC3CCN(Cc4ccccc4)CC3)o2)c(Cl)c1,CHEMBL6161637,250.0,0.250,active
8813,Nc1c2c(nc3c1C(c1cccc([N+](=O)[O-])c1)C1=C(O3)C...,CHEMBL6188525,19600.0,19.600,inactive


In [22]:
# save the DataFrame to CSV file
df5.to_csv('BACE1_bioactivity_curated_data.csv',index=False)

In [28]:
import zipfile
import glob

with zipfile.ZipFile('BACE1.zip','w',compression=zipfile.ZIP_DEFLATED) as zipf:
  for file in glob.glob('*.csv'):
    zipf.write(file)
# ! zip BACE1.zip *.csv

In [41]:
import os

os.chdir("D:\\project msc 2\\QSAR\\AChE")
!dir

 Volume in drive D is New Volume
 Volume Serial Number is 9444-9510

 Directory of D:\project msc 2\QSAR\AChE

04-08-2026  11:10    <DIR>          .
03-08-2026  08:18    <DIR>          ..
04-08-2026  11:10           469,818 BACE1.zip
04-08-2026  11:01           818,648 BACE1_activities_cleaned.csv
04-08-2026  11:01         1,265,201 BACE1_activities_raw.csv
04-08-2026  11:02           892,723 BACE1_bioactivity_curated_data.csv
04-08-2026  11:17            34,253 BACE1_data_curation_part _1.ipynb
               5 File(s)      3,480,643 bytes
               2 Dir(s)  190,772,236,288 bytes free
